In [1]:
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

c:\Users\ASHUTOSH\anaconda3\envs\tf_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ASHUTOSH\anaconda3\envs\tf_env\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [7]:
import feedparser

def get_arxiv(query):
    papers = []
    url = f'http://export.arxiv.org/api/query?search_query=all:{query}&start=0&max_results=2'

    feed = feedparser.parse(url)
    papers = []
    for entry in feed.entries:
        title = entry.title
        summary = entry.summary

        authors = [author.name for author in entry.authors]

        pdf_link = ""
        for link in entry.links:
            if link.type == "application/pdf":
                pdf_link = link.href

        abstract_link = entry.id

        papers.append({
        "Title": title,
        "Authors": authors,
        "Summary": summary,
        "PDF": pdf_link,
        "Abstract": abstract_link
        })

    return papers

In [8]:
p = get_arxiv("MachineLearning")

for pr in p:
    print(pr)
    print("-" * 50)

{'Title': 'From Cutting Planes Algorithms to Compression Schemes and Active Learning', 'Authors': ['Liva Ralaivola', 'Ugo Louche'], 'Summary': 'Cutting-plane methods are well-studied localization(and optimization) algorithms. We show that they provide a natural framework to perform machinelearning ---and not just to solve optimization problems posed by machinelearning--- in addition to their intended optimization use. In particular, theyallow one to learn sparse classifiers and provide good compression schemes.Moreover, we show that very little effort is required to turn them intoeffective active learning methods. This last property provides a generic way todesign a whole family of active learning algorithms from existing passivemethods. We present numerical simulations testifying of the relevance ofcutting-plane methods for passive and active learning tasks.', 'PDF': 'https://arxiv.org/pdf/1508.02986v1', 'Abstract': 'http://arxiv.org/abs/1508.02986v1'}
--------------------------------

In [4]:
load_dotenv()

True

In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
)

In [9]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("https://arxiv.org/pdf/2010.13852v1")
docs = loader.load()

# print(docs)

In [10]:
spliiter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = spliiter.split_documents(docs)

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="allenai/specter2_base"
)

vectore_store = FAISS.from_documents(chunks, embeddings)


In [12]:
retriever = vectore_store.as_retriever(search_type = 'similarity', search_kwargs={'k':3})

In [ ]:
# retriever.invoke("Perceptron-based Localization Algorithm")

[Document(id='11fe712d-370b-4091-b15a-23e1ced03db7', metadata={'producer': 'pdfTeX-1.40.12', 'creator': 'LaTeX with hyperref package', 'creationdate': '2018-08-07T08:53:31+00:00', 'author': '', 'keywords': '', 'moddate': '2018-08-07T08:53:31+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.3-1.40.12 (TeX Live 2011) kpathsea version 6.0.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'https://arxiv.org/pdf/1508.02986v1', 'total_pages': 14, 'page': 2, 'page_label': '3'}, page_content='each step and query the normalized solution wt = ˜wt/∥ ˜wt∥2.\nIntuitively, we may expect ˜wt+1 to be ‘close’ to ˜wt because\nCt+1 is essentially the intersection of Ct with a cutting plane\nand much of the geometry of Ct might be preserved. According\nto this intuition, ˜wt should be a good starting point for the\nPerceptron algorithm to be run and to have it output ˜wt+1.\nAlgorithm 3 implements that idea, and reuses the last query\npoint as an initialization vector for the P

In [16]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template = """
        You are a expert in research field, your job is to answer the user query on the research paper based on the context provided to you. 
        context: {context}
        query: {question}
    """,
    input_variables = ['context', 'question']
)

In [17]:
question = "Intrusion detectionframework for IoT"
retrieved_docs = retriever.invoke(question)
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'which would help improve the detection accuracy. \nAlso, contrasting the cost of searching for a suitable \nsubset of the features to the performance of the \ndetection models needs to be evaluated.   \n \nThus, this study recommends the deployment of the hybrid \napproach for botnet IoT detection, as a  hybrid IDS that \ncombines signature-based and behavioral detection may \nstand a better chance at detection IoT botnets in a timely \nfashion. In such a system, the signature -based detection \nwould help quickly identify previ ously-seen attacks, while \nthe behavioral detection component would address remaining \nzero-day/unrecognizable attacks. This implies that a feedback \nsubsystem must be incorporated with such IDS to help it \n“patternize” newly -detected attacks and add them to t he \npattern-library that would be associated with the signature -\nbased component of this IDS. \nIII. CONCLUSION \n IoT devices and networks have become an integral part\n\n"DDoS" rule, and all re

In [18]:
final_prompt = prompt.invoke({"context": context_text, "question": question})
response = llm.invoke(final_prompt)
print(response.content)

**Intrusion‑Detection Framework for IoT – A Hybrid, Feedback‑Driven Architecture**

Below is a concise, research‑oriented description of an IoT‑focused intrusion‑detection framework that synthesises the key ideas presented in the supplied excerpts. It is organised into the major building blocks, the data‑flow, and the practical considerations that make the design suitable for resource‑constrained IoT environments.

---

## 1. High‑Level Architecture  

```
                +-------------------+
                |   IoT Devices /   |
                |   Edge Gateways   |
                +--------+----------+
                         |
                         | (network traffic, system logs, sensor data)
                         v
+-----------------------------------------------------------+
|                     Hybrid IDS Core                        |
|  -------------------------------------------------------  |
|  |  Signature‑Based Engine  |  Behavioral Engine  |      |
|  |  (C5.0 de